# Text2SQL with LangChain —  Notebook
### SQL Chains, Agents, Schema RAG | `qwen3.5:4b` + `qwen3-embedding:0.6b`


In [18]:
# ── Cell 1: Install Dependencies ──────────────────────────────────────────────

import subprocess, sys

packages = [
    "langchain", "langchain-community", "langchain-ollama",
    "langchain-chroma", "langchain-classic", "chromadb",
    "ollama", "pandas", "matplotlib", "sqlalchemy",
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    print(f"{'OK' if result.returncode == 0 else 'FAIL'} {pkg}")

print("\nAll LangChain dependencies installed!")

OK langchain
OK langchain-community
OK langchain-ollama
OK langchain-chroma
OK langchain-classic
OK chromadb
OK ollama
OK pandas
OK matplotlib
OK sqlalchemy

All LangChain dependencies installed!


In [19]:
# ── Cell 2: Configuration + Connectivity Check ────────────────────────────────
import ollama

LLM_MODEL   = "qwen3.5:4b"
EMBED_MODEL = "qwen3-embedding:0.6b"
DB_PATH     = "./swiggy_demo.db"

print("Checking Ollama models...")
for model in [LLM_MODEL, EMBED_MODEL]:
    try:
        if "embedding" in model:
            emb = ollama.embeddings(model=model, prompt="test")
            print(f"OK {model} ({len(emb.embedding)}-dim)")
        else:
            resp = ollama.chat(model=model, messages=[{"role": "user", "content": "OK"}], think=False)
            print(f"OK {model}")
    except Exception as e:
        print(f"FAIL {model}: {e}  ->  ollama pull {model}")

print()
try:
    import langchain
    print(f"LangChain version: {langchain.__version__}")
except Exception as e:
    print(f"LangChain import error: {e}")

Checking Ollama models...
OK qwen3.5:4b
OK qwen3-embedding:0.6b (1024-dim)

LangChain version: 1.3.9


In [ ]:
# ── Cell 3: Build / Verify Database ───────────────────────────────────────────


import sqlite3, pandas as pd, random, os
from datetime import datetime, timedelta

if not os.path.exists(DB_PATH):
    print("Database not found — creating...")
    conn = sqlite3.connect(DB_PATH)
    cur  = conn.cursor()
    cur.executescript("""
    CREATE TABLE restaurants (id INTEGER PRIMARY KEY, name TEXT NOT NULL,
        city TEXT NOT NULL, cuisine TEXT NOT NULL, rating REAL, is_active INTEGER DEFAULT 1);
    CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT NOT NULL,
        city TEXT NOT NULL, email TEXT UNIQUE NOT NULL, joined_date TEXT NOT NULL);
    CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, restaurant_id INTEGER,
        total_amount REAL NOT NULL, status TEXT NOT NULL, payment_mode TEXT NOT NULL,
        created_at TEXT NOT NULL, delivered_at TEXT);
    CREATE TABLE order_items (id INTEGER PRIMARY KEY, order_id INTEGER,
        item_name TEXT NOT NULL, quantity INTEGER NOT NULL, unit_price REAL NOT NULL);
    CREATE TABLE reviews (id INTEGER PRIMARY KEY, order_id INTEGER, restaurant_id INTEGER,
        customer_id INTEGER, rating INTEGER NOT NULL, comment TEXT, created_at TEXT NOT NULL);
    """)
    random.seed(42)
    cities = ["Mumbai","Delhi","Bangalore","Hyderabad","Chennai"]
    cuisines = ["Indian","Chinese","Italian","Fast Food","South Indian"]
    rnames = ["Spice Garden","Dragon Palace","Pizza Corner","Burger Hub","Dosa Express",
              "Biryani House","Noodle World","Tandoor Nights","The Curry Bowl","Pasta Primo",
              "Wok & Roll","Masala Twist","Street Bites","Royal Kitchen","Fresh Box"]
    cur.executemany("INSERT INTO restaurants VALUES(?,?,?,?,?,?)",
        [(i+1,n,random.choice(cities),random.choice(cuisines),round(random.uniform(2.5,5.0),1),1)
         for i,n in enumerate(rnames)])
    custs = [(i+1,f"Customer_{i+1}",random.choice(cities),f"u{i+1}@x.com",
              (datetime.now()-timedelta(days=random.randint(30,730))).strftime("%Y-%m-%d"))
             for i in range(200)]
    cur.executemany("INSERT INTO customers VALUES(?,?,?,?,?)", custs)
    statuses=["delivered"]*75+["cancelled"]*20+["placed"]*5
    payments=["UPI"]*50+["Card"]*25+["COD"]*15+["Wallet"]*10
    ords=[]
    for i in range(1200):
        s=random.choice(statuses); p=random.choice(payments)
        ct=(datetime.now()-timedelta(days=random.randint(0,60),minutes=random.randint(0,1440))).strftime("%Y-%m-%d %H:%M:%S")
        dt=(datetime.strptime(ct,"%Y-%m-%d %H:%M:%S")+timedelta(minutes=random.randint(20,90))).strftime("%Y-%m-%d %H:%M:%S") if s=="delivered" else None
        ords.append((i+1,random.randint(1,200),random.randint(1,15),round(random.uniform(150,1200),2),s,p,ct,dt))
    cur.executemany("INSERT INTO orders VALUES(?,?,?,?,?,?,?,?)", ords)
    foods=["Biryani","Butter Chicken","Dal Makhani","Naan","Paneer Tikka","Fried Rice","Noodles","Pasta","Pizza","Burger","Dosa"]
    items,iid=[],1
    for o in ords:
        for _ in range(random.randint(1,4)):
            items.append((iid,o[0],random.choice(foods),random.randint(1,3),round(random.uniform(50,350),2))); iid+=1
    cur.executemany("INSERT INTO order_items VALUES(?,?,?,?,?)", items[:5000])
    deliv=[o for o in ords if o[4]=="delivered"]
    revs=[(i+1,o[0],o[2],o[1],random.randint(2,5),random.choice(["Great!","Good.",None]),o[6])
          for i,o in enumerate(random.sample(deliv,min(600,len(deliv))))]
    cur.executemany("INSERT INTO reviews VALUES(?,?,?,?,?,?,?)", revs)
    conn.commit(); conn.close()
    print("Database created!")
else:
    print(f"Database found: {DB_PATH}")
    c = sqlite3.connect(DB_PATH)
    for t in ["restaurants","customers","orders","order_items","reviews"]:
        print(f"  {t:<20}: {c.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]:>5} rows")
    c.close()

Database found: ./swiggy_demo.db
  restaurants         :    15 rows
  customers           :   200 rows
  orders              :  1200 rows
  order_items         :  2968 rows
  reviews             :   600 rows


---
# Section 1 — LangChain SQL Ecosystem

## The Components

```
LangChain SQL Toolkit
  SQLDatabase              <- wraps SQLite/Postgres/MySQL
    .get_table_info()      <- auto-extracts schema (no manual SCHEMA_DESCRIPTION!)
    .run(sql)              <- executes SQL safely

  create_sql_query_chain   <- LLM + DB -> SQL query

  QuerySQLDatabaseTool     <- LangChain tool: executes SQL on a DB

  FewShotPromptTemplate    <- structured few-shot examples

  OllamaEmbeddings         <- local embeddings via Ollama
  Chroma.from_texts()      <- vectorstore for Schema RAG

  create_sql_agent         <- ReAct agent with SQL tools
```

## LangChain vs Raw Ollama

| Aspect | Raw Ollama (deep-dive NB) | LangChain |
|--------|--------------------------|-----------|
| Schema extraction | Manual SCHEMA_DESCRIPTION | Auto via `db.get_table_info()` |
| SQL execution | `pd.read_sql()` | `QuerySQLDatabaseTool` |
| Lines of code | ~200 | ~50 |
| Control | Full | Structured |
| Agents | Build yourself | Built-in |

## When to Use Which?

- **Raw Ollama** — Learning, maximum control, custom security
- **LangChain** — Fast prototyping, team uses LangChain, need agents
- **Both** — Production: LangChain for speed + custom security on top

---
# Section 2 — SQLDatabase Utility

## What SQLDatabase Does

LangChain wraps SQLAlchemy to:
1. **Auto-extract schema** — `db.get_table_info()` reads PRAGMA table_info for you
2. **Execute SQL safely** — `db.run(sql)` with configurable row limits
3. **Select tables** — include/exclude specific tables from schema

```python
db = SQLDatabase.from_uri("sqlite:///swiggy_demo.db")
print(db.get_table_info())   # Auto-generated schema — no manual work!
print(db.run("SELECT COUNT(*) FROM orders;"))
```

## Why This Matters

In the deep-dive notebook, we spent Cell 5 manually writing `SCHEMA_DESCRIPTION`.
`SQLDatabase.get_table_info()` does this automatically, including sample rows.

Trade-off: auto-extraction gives less context (no enum values, no FK comments).
For better accuracy, you still need to provide the manual schema string from Cell 5.

In [21]:
# ── Cell 4: SQLDatabase Utility ───────────────────────────────────────────────
# SQLDatabase wraps SQLAlchemy to auto-extract schema and execute SQL.
# HOW: from_uri() takes a SQLAlchemy connection string.

from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri(
    f"sqlite:///{DB_PATH}",
    sample_rows_in_table_info=2,   # include 2 sample rows in schema description
    include_tables=[               # explicitly list which tables to use
        "restaurants", "customers", "orders", "order_items", "reviews"
    ]
)

print("SQLDatabase connected!")
print(f"  Dialect : {db.dialect}")
print(f"  Tables  : {db.get_usable_table_names()}")
print()
print("Auto-generated schema (first 1500 chars):")
print("-" * 60)
print(db.get_table_info()[:1500])
print()
print("Compare this to our manual SCHEMA_DESCRIPTION from the deep-dive notebook.")
print("Notice: auto-extraction is less detailed (no enum values, no FK comments).")
print("For production, combine both: auto schema + manual comments.")

SQLDatabase connected!
  Dialect : sqlite
  Tables  : ['customers', 'order_items', 'orders', 'restaurants', 'reviews']

Auto-generated schema (first 1500 chars):
------------------------------------------------------------

CREATE TABLE customers (
	id INTEGER, 
	name TEXT NOT NULL, 
	city TEXT NOT NULL, 
	email TEXT NOT NULL, 
	joined_date TEXT NOT NULL, 
	PRIMARY KEY (id), 
	UNIQUE (email)
)

/*
2 rows from customers table:
id	name	city	email	joined_date
1	Customer_1	Chennai	user1@example.com	2024-08-16
2	Customer_2	Chennai	user2@example.com	2025-05-16
*/


CREATE TABLE order_items (
	id INTEGER, 
	order_id INTEGER, 
	item_name TEXT NOT NULL, 
	quantity INTEGER NOT NULL, 
	unit_price REAL NOT NULL, 
	PRIMARY KEY (id), 
	FOREIGN KEY(order_id) REFERENCES orders (id)
)

/*
2 rows from order_items table:
id	order_id	item_name	quantity	unit_price
1	1	Idli	3	188.86
2	2	Noodles	1	74.99
*/


CREATE TABLE orders (
	id INTEGER, 
	customer_id INTEGER, 
	restaurant_id INTEGER, 
	total_amount REA

In [22]:
# ── Cell 5: SQLDatabase.run() ─────────────────────────────────────────────────
# db.run() executes SQL and returns a STRING result (not a DataFrame).
# HOW: It uses SQLAlchemy under the hood, with a built-in row limit (default 100).

test_queries = [
    "SELECT COUNT(*) as total_orders FROM orders;",
    "SELECT payment_mode, COUNT(*) as count FROM orders GROUP BY payment_mode;",
    "SELECT name, city, rating FROM restaurants LIMIT 5;",
]

print("Testing SQLDatabase.run():")
print("=" * 60)
for sql in test_queries:
    print(f"\nSQL: {sql}")
    result = db.run(sql)
    print(f"Result (string): {result}")
    print()

# Note: db.run() returns a string — if you need a DataFrame, use pd.read_sql()

Testing SQLDatabase.run():

SQL: SELECT COUNT(*) as total_orders FROM orders;
Result (string): [(1200,)]


SQL: SELECT payment_mode, COUNT(*) as count FROM orders GROUP BY payment_mode;
Result (string): [('COD', 188), ('Card', 301), ('UPI', 584), ('Wallet', 127)]


SQL: SELECT name, city, rating FROM restaurants LIMIT 5;
Result (string): [('Spice Garden', 'Mumbai', 4.4), ('Dragon Palace', 'Delhi', 2.8), ('Pizza Corner', 'Mumbai', 2.7), ('Burger Hub', 'Hyderabad', 2.6), ('Dosa Express', 'Delhi', 3.8)]



---
# Section 3 — create_sql_query_chain

## The Simplest SQL Chain

`create_sql_query_chain` creates a chain that:
1. Gets schema automatically from `SQLDatabase`
2. Builds a prompt (schema + question)
3. Calls the LLM
4. Returns the SQL string

```python
chain = create_sql_query_chain(llm, db)
sql   = chain.invoke({"question": "How many orders today?"})
```

## Under the Hood

LangChain sends this prompt internally:
```
You are a SQLite expert. Given an input question, create a
syntactically correct SQLite query to run...

Here is the relevant table info:
{table_info}

Question: {input}
SQLQuery:
```

## The Output Format Problem

The chain returns:
```
Question: How many orders today?
SQLQuery: SELECT COUNT(*) FROM orders WHERE DATE(created_at) = DATE('now')
```

We need to strip the `SQLQuery:` prefix. `clean_sql()` handles this and also removes
`<think>` tags, markdown fences, and trailing text.

In [23]:
# ── Cell 6: create_sql_query_chain ───────────────────────────────────────────
# LangChain wraps schema extraction + prompt building + LLM call in 3 lines.
# HOW: ChatOllama with think=False — uses /api/chat which properly supports think.
#      OllamaLLM uses /api/generate where think=False has NO effect.

import re
from langchain_ollama import ChatOllama
from langchain_classic.chains.sql_database.query import create_sql_query_chain

# Initialize LLM using ChatOllama (not OllamaLLM!)
# WHY ChatOllama:
#   OllamaLLM  -> /api/generate  -> think=False IGNORED  -> empty SQL output
#   ChatOllama -> /api/chat      -> think=False WORKS     -> clean SQL output
# This is the single most important configuration for qwen3.5:4b.
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0,     # deterministic output — same question = same SQL
    num_predict=512,   # max tokens to generate
    think=False,       # prevents thinking-mode output (works with /api/chat)
)

# Create the SQL chain — 1 line
sql_chain = create_sql_query_chain(llm, db)


def clean_sql(text: str) -> str:
    """Extract clean SQL from any LLM or LangChain output format.

    Handles:
    - LangChain prefix:    "Question: ...\nSQLQuery: SELECT ..."
    - Think tags:          "<think>...</think> SELECT ..."
    - Markdown fences:     "```sql\nSELECT ...\n```"
    - No trailing semicol  (common with smaller models)
    """
    original = text
    # Remove <think> blocks
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    # Remove markdown fences
    text = re.sub(r"```sql\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```\s*", "", text)
    # Strip LangChain "SQLQuery:" prefix — the main difference from deep-dive extract_sql
    lc_match = re.search(r"SQLQuery:\s*(SELECT\b.+)", text, re.DOTALL | re.IGNORECASE)
    if lc_match:
        text = lc_match.group(1)
    # With semicolon (primary)
    m = re.search(r"(SELECT\b.+?;)", text, re.DOTALL | re.IGNORECASE)
    if m:
        return m.group(1).strip()
    # Without semicolon (fallback)
    m = re.search(r"(SELECT\b.+)", text, re.DOTALL | re.IGNORECASE)
    if m:
        return m.group(1).strip()
    # Safety net: SQL was entirely inside <think> block
    for block in reversed(re.findall(r"<think>(.*?)</think>", original, re.DOTALL)):
        m = re.search(r"(SELECT\b.+)", block, re.DOTALL | re.IGNORECASE)
        if m:
            return m.group(1).strip()
    return text.strip()


print(f"SQL chain created: LLM={LLM_MODEL}, ChatOllama, think=False")
print()

# Test
test_q = "How many total orders are there?"
raw    = sql_chain.invoke({"question": test_q})
sql    = clean_sql(raw)
print(f"Q            : {test_q}")
print(f"Raw output   : {raw.strip()[:120]}")
print(f"Cleaned SQL  : {sql}")
print()
print("Result:", db.run(sql))


SQL chain created: LLM=qwen3.5:4b, ChatOllama, think=False

Q            : How many total orders are there?
Raw output   : Question: How many total orders are there?
SQLQuery: SELECT COUNT(*) FROM "orders"
Cleaned SQL  : SELECT COUNT(*) FROM "orders"

Result: [(1200,)]


In [24]:
# ── Cell 7: Test the SQL Chain ────────────────────────────────────────────────
# Validate that clean_sql() correctly strips the LangChain prefix.

questions = [
    "How many restaurants are there per city?",
    "What is the average order value?",
    "How many orders have been cancelled?",
]

print("SQL Chain Test")
print("=" * 60)
for q in questions:
    raw_sql = sql_chain.invoke({"question": q})
    sql = clean_sql(raw_sql)
    print(f"\nQ: {q}")
    print(f"SQL: {sql}")
    try:
        result = db.run(sql)
        print(f"Result: {result}")
    except Exception as e:
        print(f"Error: {e}")

SQL Chain Test

Q: How many restaurants are there per city?
SQL: SELECT COUNT(*) AS restaurant_count, "city" FROM "restaurants" GROUP BY "city" ORDER BY "restaurant_count" DESC LIMIT 5;
Result: [(5, 'Mumbai'), (3, 'Hyderabad'), (3, 'Chennai'), (2, 'Delhi'), (2, 'Bangalore')]

Q: What is the average order value?
SQL: SELECT AVG("total_amount") FROM "orders";
Result: [(674.4294,)]

Q: How many orders have been cancelled?
SQL: SELECT COUNT(*) FROM "orders" WHERE "status" = 'cancelled'
Result: [(267,)]


---
# Section 4 — Full QA Chain: Question -> Answer

## The Complete Flow

```
Question
    |
    v [create_sql_query_chain]
    SQL string
    |
    v [clean_sql()]  <- strips LangChain prefix, <think> tags, fences
    Clean SQL
    |
    v [QuerySQLDataBaseTool]
    Result string
    |
    v [Answer prompt + LLM]
    Natural language answer
```

## LCEL — LangChain Expression Language

LCEL uses the `|` operator to chain components (like Unix pipes):

```python
chain = (
    RunnablePassthrough.assign(query = write_query | clean_sql_runnable)
    .assign(result = itemgetter("query") | execute_query)
    | answer_prompt
    | llm
    | StrOutputParser()
)
```

Each component passes its output as input to the next.

## Why `clean_sql` as a Runnable?

`clean_sql` is a plain Python function. To use it in an LCEL pipe, we wrap it
with `RunnableLambda(clean_sql)` — this makes it a composable chain step.

In [25]:
# ── Cell 8: Full QA Chain ─────────────────────────────────────────────────────
# End-to-end chain: question -> SQL -> execute -> natural language answer.
# HOW: LCEL pipes (|) compose 5 components into one callable chain.

from langchain_community.tools import QuerySQLDatabaseTool   # updated import (DataBase -> Database)
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from operator import itemgetter

# Component 1: Generate SQL
write_query = create_sql_query_chain(llm, db)

# Component 2: Execute SQL
# QuerySQLDatabaseTool wraps db.run() as a LangChain tool
# NOTE: class renamed from QuerySQLDataBaseTool to QuerySQLDatabaseTool in LangChain 0.3.12
execute_query = QuerySQLDatabaseTool(db=db)

# Component 3: Answer prompt — interprets raw SQL results
ANSWER_TEMPLATE = """You are a business analyst. Answer the question based on the SQL result.

Question: {question}
SQL Query: {query}
SQL Result: {result}

Answer in 1-2 sentences with specific numbers:"""

answer_prompt = PromptTemplate.from_template(ANSWER_TEMPLATE)

# Wrap clean_sql as a Runnable so it can be used in the | pipe
clean_sql_runnable = RunnableLambda(clean_sql)


def build_qa_chain():
    """Build: question -> SQL -> clean -> execute -> interpret -> answer"""
    return (
        # Step 1: Generate SQL and clean it in one .assign()
        RunnablePassthrough.assign(query=write_query | clean_sql_runnable)
        # Step 2: Execute the clean SQL
        .assign(result=itemgetter("query") | execute_query)
        # Step 3: Format result for LLM
        | answer_prompt
        # Step 4: LLM generates the answer
        | llm
        # Step 5: Extract string from LLM output
        | StrOutputParser()
    )


qa_chain = build_qa_chain()
print("Full QA Chain built!")
print("Chain steps:")
print("  1. write_query        -> raw SQL from question")
print("  2. clean_sql_runnable -> strip prefix, think tags, fences")
print("  3. execute_query      -> run clean SQL on DB")
print("  4. answer_prompt      -> format for LLM")
print("  5. llm                -> generate answer")
print("  6. StrOutputParser    -> extract string")


Full QA Chain built!
Chain steps:
  1. write_query        -> raw SQL from question
  2. clean_sql_runnable -> strip prefix, think tags, fences
  3. execute_query      -> run clean SQL on DB
  4. answer_prompt      -> format for LLM
  5. llm                -> generate answer
  6. StrOutputParser    -> extract string


In [26]:
# ── Cell 9: Test Full QA Chain ────────────────────────────────────────────────

print("Full QA Chain Test")
print("=" * 65)
for q in [
    "How many restaurants serve Indian cuisine?",
    "What is the most common payment method?",
    "Which city has the most customers?",
]:
    print(f"\nQ: {q}")
    try:
        answer = qa_chain.invoke({"question": q})
        print(f"A: {answer}")
    except Exception as e:
        print(f"Error: {e}")
        try:
            sql = clean_sql(write_query.invoke({"question": q}))
            print(f"Generated SQL: {sql}")
        except:
            pass

Full QA Chain Test

Q: How many restaurants serve Indian cuisine?
A: 

Q: What is the most common payment method?
A: 

Q: Which city has the most customers?
A: 


---
# Section 5 — Few-Shot SQL Chain with LangChain

## Why Few-Shot with LangChain?

LangChain's default SQL chain prompt is generic.
We replace it with `FewShotPromptTemplate` to inject domain examples.

## Template Structure

```
[prefix]   <- schema + rules

[example 1] Input: How many orders today?
            SQL: SELECT COUNT(*) FROM orders WHERE DATE(created_at) = DATE('now');
[example 2] Input: Top 5 restaurants by GMV?
            SQL: SELECT r.name, SUM(o.total_amount)...
...

[suffix]   Input: {user_question}
           SQL:
```

The LLM sees the pattern and continues it with correct SQL.

## `FewShotPromptTemplate` Parameters

```python
FewShotPromptTemplate(
    examples=FEW_SHOT_EXAMPLES,       # your Q+SQL pairs
    example_prompt=example_template,   # how to format each example
    prefix="Schema + rules...",        # before the examples
    suffix="Input: {input}\nSQL:",     # after the examples
    input_variables=["input"],
)
```

In [27]:
# ── Cell 10: Few-Shot SQL Chain ────────────────────────────────────────────────
# FewShotPromptTemplate structures examples in a way LangChain chains understand.
# HOW: Build prefix (schema + rules) + examples + suffix (question placeholder).

from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate as PT
from langchain_core.output_parsers import StrOutputParser

SCHEMA_INFO = db.get_table_info()  # Auto-extracted schema from SQLDatabase

FEW_SHOT_EXAMPLES = [
    {
        "input": "How many orders were placed today?",
        "query": "SELECT COUNT(*) AS orders_today FROM orders WHERE DATE(created_at) = DATE('now');"
    },
    {
        "input": "Top 5 restaurants by total GMV in last 30 days?",
        "query": (
            "SELECT r.name, ROUND(SUM(o.total_amount),2) AS total_gmv "
            "FROM orders o JOIN restaurants r ON o.restaurant_id = r.id "
            "WHERE o.status = 'delivered' AND DATE(o.created_at) >= DATE('now','-30 days') "
            "GROUP BY r.name ORDER BY total_gmv DESC LIMIT 5;"
        )
    },
    {
        "input": "Cancellation rate by payment method?",
        "query": (
            "SELECT payment_mode, "
            "ROUND(100.0*SUM(CASE WHEN status='cancelled' THEN 1 ELSE 0 END)/COUNT(*),2) AS cancel_pct "
            "FROM orders GROUP BY payment_mode ORDER BY cancel_pct DESC;"
        )
    },
    {
        "input": "Average delivery time in minutes by city?",
        "query": (
            "SELECT r.city, "
            "ROUND(AVG((julianday(o.delivered_at)-julianday(o.created_at))*24*60),1) AS avg_min "
            "FROM orders o JOIN restaurants r ON o.restaurant_id = r.id "
            "WHERE o.status='delivered' AND o.delivered_at IS NOT NULL "
            "GROUP BY r.city ORDER BY avg_min;"
        )
    },
    {
        "input": "Top 3 most ordered food items?",
        "query": "SELECT item_name, SUM(quantity) AS total_ordered FROM order_items GROUP BY item_name ORDER BY total_ordered DESC LIMIT 3;"
    },
]

# Format for each example: how LangChain renders one Q+SQL pair
example_template = PT(
    input_variables=["input", "query"],
    template="Input: {input}\nSQL: {query}"
)

FEW_SHOT_PREFIX = f"""You are a SQLite expert for a food delivery app. Generate accurate SQL.

Database schema:
{SCHEMA_INFO}

SQLite dates: DATE('now'), DATE('now','-N days'), strftime('%Y-%m', col)
Delivery time: (julianday(delivered_at)-julianday(created_at))*24*60
JOIN: orders JOIN restaurants ON orders.restaurant_id = restaurants.id

Rules: Return ONLY SQL, no markdown, end with semicolon.

Example Q&A pairs:"""

# Build the few-shot prompt template
few_shot_prompt = FewShotPromptTemplate(
    examples=FEW_SHOT_EXAMPLES,
    example_prompt=example_template,
    prefix=FEW_SHOT_PREFIX,
    suffix="Input: {input}\nSQL:",
    input_variables=["input"],
)

# Build the chain: prompt | llm | parse output
fewshot_chain = few_shot_prompt | llm | StrOutputParser()

print(f"Few-shot chain built: {len(FEW_SHOT_EXAMPLES)} examples")
print()
print("Testing:")
for q in ["Which city generates the most revenue?", "What percentage of orders use COD?"]:
    raw = fewshot_chain.invoke({"input": q})
    sql = clean_sql(raw)
    print(f"\nQ: {q}")
    print(f"SQL: {sql}")
    try:
        print(f"Result: {db.run(sql)}")
    except Exception as e:
        print(f"Error: {e}")

Few-shot chain built: 5 examples

Testing:

Q: Which city generates the most revenue?
SQL: 
Result: 

Q: What percentage of orders use COD?
SQL: 
Result: 


---
# Section 6 — Schema RAG with LangChain

## The Problem (Recap)

- 5 tables -> ~500 tokens: fine
- 500 tables -> ~50,000 tokens: context overflow

## LangChain Components

```
OFFLINE:
Table descriptions -> OllamaEmbeddings(qwen3-embedding:0.6b)
                   -> Chroma.from_texts()  (vectorstore)

ONLINE (per query):
Question -> OllamaEmbeddings -> Chroma.similarity_search() -> top-3 docs
         -> extract table names -> inject schemas -> LLM -> SQL
```

| Component | Role |
|-----------|------|
| `OllamaEmbeddings` | Embed text with qwen3-embedding:0.6b locally |
| `Chroma.from_texts()` | Create vectorstore from descriptions |
| `.similarity_search()` | Return most relevant docs for a query |

## Key Difference from Deep-Dive Notebook

In the deep-dive notebook, we built a custom `Qwen3Embedder` class for ChromaDB.
In LangChain, `OllamaEmbeddings` handles this automatically and integrates
directly with `Chroma` — no custom class needed.

In [28]:
# ── Cell 11: Schema RAG — Build Vectorstore ───────────────────────────────────
# Same RAG pattern as deep-dive but using LangChain Chroma integration.
# HOW: OllamaEmbeddings + Chroma.from_texts() — no custom embedder class needed.

from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

# Initialize the embedding model
# WHY OllamaEmbeddings: wraps ollama.embeddings() in LangChain's Embeddings interface
embeddings = OllamaEmbeddings(model=EMBED_MODEL)

print(f"Testing {EMBED_MODEL}...")
test_emb = embeddings.embed_query("food delivery orders")
print(f"Embedding dimension: {len(test_emb)}")
print()

# Semantic table descriptions — what business questions each table answers
TABLE_DESCRIPTIONS = [
    {
        "text": "Table 'restaurants' stores restaurant information: name, city (Mumbai/Delhi/Bangalore/Hyderabad/Chennai), cuisine (Indian/Chinese/Italian/Fast Food/South Indian), rating (1.0-5.0), is_active. Use for restaurant analysis, city comparison, cuisine trends.",
        "table": "restaurants"
    },
    {
        "text": "Table 'customers' stores customer records: name, city, email, joined_date. Use for customer counts, cohort analysis, geographic distribution.",
        "table": "customers"
    },
    {
        "text": "Table 'orders' is the primary transactions table: customer_id, restaurant_id, total_amount (GMV), status (delivered/cancelled/placed), payment_mode (UPI/Card/COD/Wallet), created_at, delivered_at. Use for revenue, cancellation rates, payment trends, date-based analysis.",
        "table": "orders"
    },
    {
        "text": "Table 'order_items' contains individual food items per order: order_id, item_name, quantity, unit_price. Use for popular items, menu analysis, most ordered dishes.",
        "table": "order_items"
    },
    {
        "text": "Table 'reviews' stores customer feedback: order_id, restaurant_id, customer_id, rating (1-5), comment, created_at. Use for satisfaction scores, restaurant quality, review volume.",
        "table": "reviews"
    },
]

texts     = [d["text"] for d in TABLE_DESCRIPTIONS]
metadatas = [{"table": d["table"]} for d in TABLE_DESCRIPTIONS]

# Chroma.from_texts() embeds all texts and stores in memory
# No collection_name -> ephemeral (avoids duplicate docs on re-run)
print(f"Building Schema RAG with {EMBED_MODEL}...")
schema_store = Chroma.from_texts(
    texts=texts,
    metadatas=metadatas,
    embedding=embeddings,
)
print(f"Indexed {len(texts)} table descriptions")
print(f"Collection size: {schema_store._collection.count()}")

Testing qwen3-embedding:0.6b...
Embedding dimension: 1024

Building Schema RAG with qwen3-embedding:0.6b...
Indexed 5 table descriptions
Collection size: 10


In [29]:
# ── Cell 12: Schema RAG — Retriever + SQL Generation ─────────────────────────
# At query time, find the 3 most relevant tables and inject only those schemas.
# HOW: Chroma.similarity_search() -> extract table names -> look up full schemas.

FULL_SCHEMA_MAP = {
    "restaurants": "CREATE TABLE restaurants (id PK, name TEXT, city TEXT -- Mumbai/Delhi/Bangalore/Hyderabad/Chennai, cuisine TEXT, rating REAL -- 1.0-5.0, is_active INTEGER -- 1=open)",
    "customers":   "CREATE TABLE customers (id PK, name TEXT, city TEXT, email TEXT, joined_date TEXT -- YYYY-MM-DD)",
    "orders":      "CREATE TABLE orders (id PK, customer_id -- FK->customers.id, restaurant_id -- FK->restaurants.id, total_amount REAL -- GMV rupees, status TEXT -- delivered/cancelled/placed, payment_mode TEXT -- UPI/Card/COD/Wallet, created_at TEXT -- YYYY-MM-DD HH:MM:SS, delivered_at TEXT)",
    "order_items": "CREATE TABLE order_items (id PK, order_id -- FK->orders.id, item_name TEXT, quantity INTEGER, unit_price REAL -- rupees)",
    "reviews":     "CREATE TABLE reviews (id PK, order_id -- FK->orders.id, restaurant_id -- FK->restaurants.id, customer_id -- FK->customers.id, rating INTEGER -- 1-5, comment TEXT, created_at TEXT)",
}


def retrieve_schemas_for_question(question: str, k: int = 3) -> tuple:
    """Retrieve top-k most relevant table schemas.
    
    Uses qwen3-embedding:0.6b for semantic similarity.
    Returns: (combined_schema_string, list_of_table_names)
    """
    docs = schema_store.similarity_search(question, k=k)
    tables_found = [doc.metadata["table"] for doc in docs]
    schema_text  = "\n\n".join(
        FULL_SCHEMA_MAP[t] for t in tables_found if t in FULL_SCHEMA_MAP
    )
    return schema_text, tables_found


def generate_sql_with_rag(question: str) -> tuple:
    """Generate SQL injecting only retrieved table schemas."""
    schema_text, tables_used = retrieve_schemas_for_question(question)
    prompt_text = f"""You are a SQLite expert. Generate a SQL query.

RELEVANT TABLE SCHEMAS:
{schema_text}

SQLite dates: DATE('now'), DATE('now','-30 days'), strftime('%Y-%m', col)
JOIN: orders JOIN restaurants ON orders.restaurant_id = restaurants.id
Rules: SELECT only, return ONLY SQL, end with semicolon.

Question: {question}
SQL:"""
    from langchain_core.prompts import PromptTemplate as PT2
    from langchain_core.output_parsers import StrOutputParser
    chain = PT2.from_template("{text}") | llm | StrOutputParser()
    raw   = chain.invoke({"text": prompt_text})
    return clean_sql(raw), tables_used


# Test
print("Schema RAG Test")
print("=" * 65)
for q in [
    "What is the average review rating per restaurant?",
    "Which food items are ordered most?",
    "Total GMV by payment method?",
]:
    sql, tables = generate_sql_with_rag(q)
    print(f"\nQ: {q}")
    print(f"Tables retrieved: {tables}")
    print(f"SQL: {sql}")
    try:
        print(f"Result: {db.run(sql)[:200]}")
    except Exception as e:
        print(f"Error: {e}")

Schema RAG Test

Q: What is the average review rating per restaurant?
Tables retrieved: ['restaurants', 'restaurants', 'reviews']
SQL: 
Result: 

Q: Which food items are ordered most?
Tables retrieved: ['order_items', 'order_items', 'orders']
SQL: 
Result: 

Q: Total GMV by payment method?
Tables retrieved: ['orders', 'orders', 'order_items']
SQL: 
Result: 


---
# Section 7 — SQL Agent

## What Is a SQL Agent?

A SQL Agent is an LLM that can call SQL tools in a loop:

```
Agent thinks -> calls a tool -> gets result -> thinks again -> ...
```

## Tools Available to the SQL Agent

| Tool | What It Does |
|------|-------------|
| `sql_db_list_tables` | List all tables in the DB |
| `sql_db_schema` | Get schema for specific tables |
| `sql_db_query` | Run a SQL query |
| `sql_db_query_checker` | Validate SQL before running |

## Agent vs Chain

| | SQL Chain | SQL Agent |
|-|-----------|-----------|
| **Steps** | One-shot: question -> SQL -> done | Multi-step loop |
| **Best for** | Simple, predictable questions | Complex, multi-step questions |
| **Speed** | Fast | Slower (multiple LLM calls) |
| **Control** | Predictable | Less predictable |

## The ReAct Pattern

```
Thought: "I need to find which table has order data..."
Action: sql_db_list_tables
Observation: restaurants, customers, orders, order_items, reviews
Thought: "Let me check the orders schema..."
Action: sql_db_schema(orders)
Observation: id, customer_id, total_amount, status...
Thought: "Now I can write the query."
Action: sql_db_query(SELECT ...)
Final Answer: ...
```

In [30]:
# ── Cell 13: SQL Agent ────────────────────────────────────────────────────────
# Agents handle complex multi-step questions chains cannot.
# HOW: SQLDatabaseToolkit provides 4 tools; create_sql_agent wires them together.

from langchain_community.agent_toolkits import create_sql_agent
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=llm)
print("SQL toolkit tools:")
for tool in toolkit.get_tools():
    print(f"  -> {tool.name}: {tool.description[:60]}...")
print()

agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True,             # show agent reasoning steps
    handle_parsing_errors=True,
    max_iterations=10,        # prevent infinite loops
)

print("SQL Agent created!")
print("The agent will:")
print("  1. List tables to understand the database")
print("  2. Check relevant table schemas")
print("  3. Generate and run SQL queries")
print("  4. Return a natural language answer")

SQL toolkit tools:
  -> sql_db_query: Input to this tool is a detailed and correct SQL query, outp...
  -> sql_db_schema: Input to this tool is a comma-separated list of tables, outp...
  -> sql_db_list_tables: Input is an empty string, output is a comma-separated list o...
  -> sql_db_query_checker: Use this tool to double check if your query is correct befor...

SQL Agent created!
The agent will:
  1. List tables to understand the database
  2. Check relevant table schemas
  3. Generate and run SQL queries
  4. Return a natural language answer


In [31]:
# ── Cell 14: Test SQL Agent ────────────────────────────────────────────────────
# Use simpler questions first — agents are slower but more autonomous.

agent_question = "What are the top 3 restaurants by number of orders?"

print(f"Agent Question: {agent_question}")
print("=" * 65)
print("(Agent reasoning steps shown below)")
print()

try:
    answer = agent.invoke({"input": agent_question})
    print()
    print("FINAL ANSWER:")
    print("-" * 40)
    print(answer.get("output", str(answer)))
except Exception as e:
    print(f"Agent error: {e}")
    print()
    print("Note: For complex multi-step reasoning, try qwen3.5:8b or larger models.")

Agent Question: What are the top 3 restaurants by number of orders?
(Agent reasoning steps shown below)



> Entering new SQL Agent Executor chain...
Thought: I should look at the tables in the database to see what I can query about restaurants and orders. Let me start by listing all available tables.

Action: sql_db_list_tables
Action Input:customers, order_items, orders, restaurants, reviewsThought: Now I can see there are 5 tables: customers, order_items, orders, restaurants, reviews. To find top 3 restaurants by number of orders, I need to look at both the "restaurants" and "orders" tables. Let me get the schema for these relevant tables first.

Action: sql_db_schema
Action Input: restaurants, orders
CREATE TABLE orders (
	id INTEGER, 
	customer_id INTEGER, 
	restaurant_id INTEGER, 
	total_amount REAL NOT NULL, 
	status TEXT NOT NULL, 
	payment_mode TEXT NOT NULL, 
	created_at TEXT NOT NULL, 
	delivered_at TEXT, 
	PRIMARY KEY (id), 
	FOREIGN KEY(restaurant_id) REFERENCES restaurant

---
# Section 8 — Security Layer for LangChain

## What LangChain Handles and What It Does Not

LangChain's `QuerySQLDataBaseTool` has a built-in row limit but does NOT:
- Block DELETE/DROP/UPDATE SQL
- Enforce SELECT-only queries
- Protect against SQL injection in the question

We wrap LangChain's execution with the same security layer from the deep-dive notebook.

## The Wrapper Pattern

```python
def safe_execute_lc(sql: str) -> str:
    safe, reason = validate_sql_lc(sql)
    if not safe:
        return f"BLOCKED: {reason}"
    return db.run(sql)  # LangChain's safe executor
```

This pattern lets you use LangChain's convenient abstractions while adding
your own security layer on top.

In [32]:
# ── Cell 15: Security Wrapper for LangChain ───────────────────────────────────
import re

FORBIDDEN = [
    r"\bDELETE\b", r"\bDROP\b",    r"\bTRUNCATE\b",
    r"\bINSERT\b", r"\bUPDATE\b",  r"\bALTER\b",
    r"\bCREATE\b", r"\bATTACH\b",  r"\bPRAGMA\b",
]


def validate_sql_lc(sql: str) -> tuple:
    """Validate SQL before executing via LangChain."""
    if not sql or not sql.strip():
        return False, "Empty SQL"
    upper = sql.strip().upper()
    if not upper.startswith("SELECT"):
        return False, f"Must start with SELECT"
    for pat in FORBIDDEN:
        if re.search(pat, upper):
            return False, f"Forbidden: {re.search(pat, upper).group()}"
    return True, "OK"


def safe_execute_lc(sql: str, max_rows: int = 100) -> str:
    """Execute SQL with validation and row limit."""
    safe, reason = validate_sql_lc(sql)
    if not safe:
        return f"BLOCKED: {reason}"
    if "LIMIT" not in sql.upper():
        sql = sql.rstrip(";").strip() + f" LIMIT {max_rows};"
    try:
        return db.run(sql)
    except Exception as e:
        return f"SQL Error: {e}"


# Test
print("Security Wrapper Tests:")
for sql in [
    "SELECT COUNT(*) FROM orders;",
    "DELETE FROM orders;",
    "SELECT * FROM customers WHERE 1=1;",
    "DROP TABLE restaurants;",
]:
    result = safe_execute_lc(sql)
    status = "OK" if not result.startswith("BLOCK") and not result.startswith("SQL Error") else "NO"
    print(f"  {status} {sql[:50]:50} -> {str(result)[:50]}")

Security Wrapper Tests:
  OK SELECT COUNT(*) FROM orders;                       -> [(1200,)]
  NO DELETE FROM orders;                                -> BLOCKED: Must start with SELECT
  OK SELECT * FROM customers WHERE 1=1;                 -> [(1, 'Customer_1', 'Chennai', 'user1@example.com',
  NO DROP TABLE restaurants;                            -> BLOCKED: Must start with SELECT


---
# Section 9 — Full Production Pipeline with LangChain

## Combining Everything

```
Question
    | [1] Schema RAG     (OllamaEmbeddings + Chroma)
    | [2] SQL Generation (few-shot prompt + dynamic schema + LLM)
    | [3] Security check (validate_sql_lc)
    | [4] Execute        (safe_execute_lc)
    | [5] Interpret      (LLM answer from raw result)
    v
{ success, answer, sql, result, tables }
```

Same logical flow as the deep-dive notebook, but using LangChain components
for schema extraction and chain composition.

In [33]:
# ── Cell 16: Full Production Pipeline ─────────────────────────────────────────
from langchain_core.prompts import PromptTemplate as PT3
from langchain_core.output_parsers import StrOutputParser


def step1_retrieve_schemas(question: str) -> tuple:
    """Schema RAG: get top-3 relevant tables."""
    return retrieve_schemas_for_question(question, k=3)


def step2_generate_sql(question: str, schema: str) -> str:
    """Generate SQL with few-shot prompt + dynamic schema."""
    examples_str = "\n".join(
        f"Q: {e['input']}\nSQL: {e['query']}"
        for e in FEW_SHOT_EXAMPLES[:3]
    )
    prompt_text = f"""You are a SQLite expert for a food delivery app.

SCHEMA:
{schema}

EXAMPLES:
{examples_str}

SQLite dates: DATE('now'), DATE('now','-N days'), strftime('%Y-%m', col)
RULES: SELECT only, return ONLY the SQL, end with semicolon.

Question: {question}
SQL:"""
    chain = PT3.from_template("{text}") | llm | StrOutputParser()
    raw   = chain.invoke({"text": prompt_text})
    return clean_sql(raw)


def step3_execute(sql: str, question: str, max_retries: int = 1) -> tuple:
    """Execute with security check + one auto-fix retry."""
    for attempt in range(max_retries + 1):
        safe, reason = validate_sql_lc(sql)
        if not safe:
            return None, f"BLOCKED: {reason}"
        result = safe_execute_lc(sql)
        if not result.startswith("SQL Error"):
            return result, sql
        if attempt < max_retries:
            fix_prompt = f"Fix this SQL error. Return ONLY the corrected SQL.\nBroken: {sql}\nError: {result}\nFixed:"
            chain2 = PT3.from_template("{t}") | llm | StrOutputParser()
            sql = clean_sql(chain2.invoke({"t": fix_prompt}))
    return None, f"Failed: {result}"


def step4_interpret(question: str, sql: str, result: str) -> str:
    """Interpret raw SQL result as a business insight."""
    prompt_text = f"""You are a business analyst. Answer the question based on the SQL result.

Question: {question}
SQL: {sql}
Result: {result}

Answer (1-2 sentences with specific numbers):"""
    chain = PT3.from_template("{t}") | llm | StrOutputParser()
    return chain.invoke({"t": prompt_text})


def lc_text_to_sql(question: str, verbose: bool = True) -> dict:
    """Full production Text2SQL pipeline using LangChain components."""
    if verbose: print(f"[1/4] Schema RAG...")
    schema, tables = step1_retrieve_schemas(question)
    if verbose: print(f"      Tables: {tables}")
    if verbose: print(f"[2/4] SQL Generation...")
    sql = step2_generate_sql(question, schema)
    if verbose: print(f"      SQL: {sql[:70]}")
    if verbose: print(f"[3/4] Execute + Security...")
    result, final_sql = step3_execute(sql, question)
    if result is None:
        return {"success": False, "error": final_sql, "question": question}
    if verbose: print(f"      Result: {str(result)[:80]}")
    if verbose: print(f"[4/4] Interpreting...")
    answer = step4_interpret(question, final_sql, result)
    return {"success": True, "answer": answer, "sql": final_sql,
            "result": result, "tables": tables}


print("LangChain production pipeline ready!")
print("Usage: result = lc_text_to_sql('your question')")

LangChain production pipeline ready!
Usage: result = lc_text_to_sql('your question')


In [34]:
# ── Cell 17: Test Full Production Pipeline ─────────────────────────────────────

for q in [
    "Which city has the highest total GMV from delivered orders?",
    "What are the top 3 most popular food items?",
    "Show restaurants with average rating above 4.0",
]:
    print("\n" + "=" * 65)
    print(f"QUESTION: {q}")
    print("=" * 65)
    result = lc_text_to_sql(q)
    print()
    if result["success"]:
        print(f"Tables: {result['tables']}")
        print(f"SQL: {result['sql']}")
        print(f"\nResult: {result['result']}")
        print(f"\nInsight: {result['answer']}")
    else:
        print(f"FAIL: {result['error']}")


QUESTION: Which city has the highest total GMV from delivered orders?
[1/4] Schema RAG...
      Tables: ['orders', 'orders', 'restaurants']
[2/4] SQL Generation...
      SQL: 
[3/4] Execute + Security...

FAIL: BLOCKED: Empty SQL

QUESTION: What are the top 3 most popular food items?
[1/4] Schema RAG...
      Tables: ['order_items', 'order_items', 'restaurants']
[2/4] SQL Generation...
      SQL: 
[3/4] Execute + Security...

FAIL: BLOCKED: Empty SQL

QUESTION: Show restaurants with average rating above 4.0
[1/4] Schema RAG...
      Tables: ['restaurants', 'restaurants', 'reviews']
[2/4] SQL Generation...
      SQL: 
[3/4] Execute + Security...

FAIL: BLOCKED: Empty SQL
